# Demo AI – từ ảnh MRI đến gợi ý bệnh theo tầng

Notebook này làm 2 việc:

- **Phần A** – giải thích hai thư mục chính của repo: `pipeline/` (nghiên cứu offline trên 515 bệnh nhân) và `ai/` (xử lý từng ca mới cho web).
- **Phần B** – tái tạo hình [`demo_0100.png`](demo_0100.png) từng bước, chỉ bằng code có sẵn trong repo.

Chạy bằng `.venv` ở gốc repo (Python 3.12). Mặc định **không cần GPU / TotalSpineSeg**: notebook dùng mask có sẵn.

## A. Bức tranh chung

```
OFFLINE – pipeline/  (chạy 1 lần trên 515 bệnh nhân, sinh bảng / hình cho báo cáo và HỌC ngưỡng)

  DICOM ─s01─► chỉ mục ─s02─► NIfTI sagittal ─(Kaggle GPU) TotalSpineSeg─► mask ─s05 QC─► s09 số đo / tầng ─┐
  Ghi chú bác sĩ (xlsx) ─s06─s07─s08─► nhãn bệnh / tầng (NLP bằng rule) ──────────────────────────────────┴─► s10 ghép ─► s11 ngưỡng
                                                                                                                           ║
ONLINE – ai/  (mỗi ca người dùng upload qua web)                                                                          ║
                                                                                                                           ▼
  ảnh NIfTI ─► segment()  (TotalSpineSeg) ─► mask ─► analyze()  (measure_case của s09 + bảng ngưỡng của s11) ─► số đo + gợi ý ─► BE ─► FE
```

- `pipeline/` **học**: từ 513 bệnh nhân dùng được, tìm xem số đo nào trên ảnh khớp với những gì bác sĩ ghi, và học ngưỡng.
- `ai/` **áp dụng**: đo một ca mới theo **đúng cùng cách đo** (`measure_case()` của s09), rồi so với bảng ngưỡng của s11. `ai/` không học gì thêm.
- Hai bên dùng chung khóa **`(patient_id, level)`** – bệnh nhân × tầng đĩa đệm (T12-L1 … L5-S1).

## A1. `pipeline/` – nghiên cứu trên bộ dữ liệu

Mọi script chạy từ thư mục `pipeline/`; đường dẫn chung ở `src/common.py`. Bảng ghi vào `outputs/tables/<nhóm>/`, hình vào `outputs/figures/<nhóm>/`.

| Bước | Script | Đầu vào | Đầu ra chính |
|---|---|---|---|
| 0 | `s00_shared_data.py --fetch` | GitHub Release | mask 515 ca + `tss_qc.csv` (khỏi chạy lại bước 2) |
| 1 | `dicom/s01_index_dicom.py` | `data/MRI_Data/01_MRI_Data/*` (DICOM) | `dicom_index.csv` – mỗi file ảnh 1 dòng |
| 1 | `dicom/s02_select_convert.py` | `dicom_index.csv` + DICOM | `outputs/nifti/sag/0001.nii.gz`, `ax/…`, `series_selected.csv` |
| 2 | `segmentation/s03_tss_pilot.py` | vài ảnh NIfTI | đo thời gian / RAM khi chạy CPU |
| 2 | `segmentation/s04_pack_for_kaggle.py` + `kaggle/tss_kaggle.ipynb` | 515 ảnh sagittal | mask 515 ca (Kaggle T4, ~18 s/ca) |
| 2 | `segmentation/s05_tss_qc.py` | mask + ảnh | `tss_qc.csv` (`qc_pass`, `use_level`) |
| 3 | `nlp/s06_gold_sample.py` | ghi chú bác sĩ | chia 435 dev / 80 gold |
| 3 | `nlp/nlp_rules.py` + `s07_nlp_extract.py --all` | ghi chú | mỗi phát hiện 1 dòng (tầng, bệnh, khẳng định / phủ định) |
| 3 | `nlp/s08_nlp_eval.py` | kết quả s07 + nhãn gold | F1, `labels_level.csv`, `labels_patient.csv` |
| 4 | `measurements/s09_level_features.py` | mask + ảnh + `tss_qc.csv` | `level_features.csv` (số đo / tầng), `patient_features.csv` |
| 5 | `analysis/s10_level_analysis.py` | số đo + nhãn NLP | AUC từng cặp số đo ↔ bệnh, ca lệch ảnh ↔ báo cáo |
| 6 | `analysis/s11_threshold_rules.py` | bảng ghép của s10 | `threshold_rules.csv`, `threshold_cv_by_level.csv`, `measure_reference.csv` |

**Bước 1 – DICOM → NIfTI.**
- s01 chỉ đọc header của mọi file:
  - Mặt cắt tính từ `ImageOrientationPatient`, không tin tên thư mục.
  - T1/T2 lấy từ tên series, kiểm chéo bằng EchoTime.
  - Đánh cờ localizer và POSDISP.
- s02 chọn **một series T2 sagittal** cho mỗi bệnh nhân, ưu tiên lần lượt:
  1. study L-SPINE;
  2. trường nhìn ≥ 250 mm;
  3. tâm cách vùng chụp axial ≤ 100 mm;
  4. nhiều lát nhất;
  5. chụp sau.

  Sau đó `build_volume()` xếp lát theo vị trí thật, bỏ lát lưu trùng, và ghi NIfTI đúng hình học (gốc, spacing, hướng).

**Bước 2 – phân đoạn.**
- TotalSpineSeg (mô hình nnU-Net huấn luyện sẵn) được dùng **nguyên bản**, không fine-tune. Mask gán mỗi voxel một nhãn: 41–45 = đốt L1–L5, 50 = xương cùng, 91–95 và 100 = đĩa T12-L1 … L5-S, 1–2 = tủy và ống sống.
- s05 kiểm tra từng ca:
  - mask cùng lưới với ảnh;
  - đủ đốt, đĩa và ống sống;
  - thứ tự đầu → chân đúng;
  - không bị vỡ mảnh;
  - không có đốt L6.
- Kết quả: 514/515 đạt, **513 dùng được theo tầng**.

**Bước 3 – NLP bằng rule.**
- Các bước xử lý:
  1. Chuẩn hóa văn bản, sửa lỗi chính tả.
  2. Tách mệnh đề.
  3. Nhận tầng bằng regex.
  4. Nhận bệnh bằng từ điển regex.
  5. Xét phủ định kiểu NegEx (`no`, `without`, `not seen`…).
  6. Lấy mức độ và bên trái/phải.
- Rule được viết trên tập dev, chấm một lần trên tập gold: **F1 0,959** ở mức tầng.

**Bước 4 – số đo (s09, `measure_case()`).**
- Đo trên 3 lát giữa (lát có nhiều ống sống nhất và 2 lát kề), trong tọa độ mm thật.
- Mỗi đĩa có trục riêng tính bằng PCA: u = trước → sau, v = chân → đầu.
- Thân đốt chỉ lấy phần **trước ống sống**, vì nhãn đốt của TotalSpineSeg gồm cả mỏm gai.
- Kết quả gồm: chiều cao đĩa, DHI, tín hiệu đĩa / dịch não tủy, tín hiệu đĩa / thân đốt, độ nhô ra sau, đường kính và diện tích ống sống, độ trượt đốt; thêm góc ưỡn cho từng bệnh nhân.

**Bước 5 – ghép ảnh × văn bản (s10).**
- Ghép số đo với nhãn NLP theo `(patient_id, level)`; tầng không được nhắc thì coi như không bệnh (nhãn yếu).
- Mỗi số đo được chuẩn hóa z **trong từng tầng**.
- Với mỗi cặp số đo ↔ bệnh, tính AUC và kiểm định Mann–Whitney.
- Tìm **ca lệch**: báo cáo không ghi bệnh nhưng số đo bất thường (nghi bỏ sót), hoặc ngược lại.

**Bước 6 – học ngưỡng (s11).**
- Mỗi cặp có một ngưỡng **Youden**: riêng cho từng tầng nếu tầng đó có ≥ 10 ca bệnh, nếu không thì dùng ngưỡng z chung.
- Đánh giá:
  - kiểm chứng chéo 5 phần × 5 lần, **chia theo bệnh nhân**, để đo độ nhạy, độ đặc hiệu, PPV trên bệnh nhân chưa thấy;
  - AUC có khoảng tin cậy 95 % bằng bootstrap theo bệnh nhân.
- Quy tắc được dùng khi cận dưới khoảng tin cậy AUC ≥ 0,70 (**35/54 quy tắc**); độ tin cậy "cao" khi ≥ 0,80.

## A2. `ai/` – xử lý một ca mới (worker của BE gọi trực tiếp)

```
ảnh upload ─► segment(input, out_dir)  ─► SegResult(mask_path, labels_present, duration_seconds)
                                       └─► SegmentationError(code, message, log_tail)   → job failed
mask ───────► analyze(input, mask)     ─► dict  = result.analysis  (qc, levels, lordosis_deg, disclaimer)
                                       └─► AnalysisError(code, message)                 → job vẫn done, mask vẫn xem được
```

**`segment.py`**
- Gọi CLI `totalspineseg INPUT OUTPUT --quiet [--iso] [--step1]` qua `subprocess`: tham số dạng list, không qua shell, nên tên file không chèn được lệnh.
- File chạy được tìm cạnh `sys.executable` (Windows: `Scripts\totalspineseg.exe`).
- Toàn bộ log ghi vào `logs/totalspineseg.log`. Khi lỗi, trả về 30 dòng cuối.
- Quá `timeout` hoặc BE tắt giữa chừng thì **kill cả cây tiến trình** (nnU-Net sinh nhiều tiến trình con) bằng `psutil`.
- Mã lỗi: `SEGMENTATION_FAILED` (CLI thoát với mã ≠ 0), `OUTPUT_NOT_FOUND` (không có đúng 1 mask trong `step2_output/`), `TIMEOUT`.
- `labels_present` = các giá trị khác 0 trong mask; FE dùng để chỉ hiện chú thích cho cấu trúc có thật.

**`analyze.py`**
1. **Kiểm tra hình học.**
   - Mask phải cùng lưới với ảnh (segment với `iso=False`), nếu không → `GEOMETRY_MISMATCH`.
   - Ảnh phải là sagittal, nếu không → `NOT_SAGITTAL`.
2. **QC nhãn** (cùng tinh thần s05): thiếu đốt hoặc đĩa thắt lưng, có đốt L6, thứ tự đầu → chân sai.
   - Không đạt → `qc.ok = false`: vẫn trả số đo nhưng **không gợi ý**, vì không chắc đánh số tầng đúng.
3. **Đo** bằng `measure_case()` của s09: 8 số đo cho mỗi tầng, cùng cách đo với 513 bệnh nhân dùng để học ngưỡng.
4. **Gợi ý**, đọc từ `pipeline/outputs/tables/analysis/`:
   - Chỉ dùng quy tắc `use = True`.
   - Mỗi (bệnh, tầng) áp dụng **một** quy tắc: ngưỡng riêng của tầng trước; nếu nhiều quy tắc thì lấy quy tắc có cận dưới khoảng tin cậy AUC cao nhất.
   - Bỏ quy tắc có PPV < 0,05. Với ngưỡng z chung, dùng PPV **tại đúng tầng đó**.
   - Mỗi gợi ý kèm: giá trị, quy tắc đã khớp, phân vị so với 513 bệnh nhân cùng tầng, độ tin cậy, độ nhạy, độ đặc hiệu, PPV khi kiểm chứng chéo.

**Đọc gợi ý thế nào.**
- Ngưỡng tái tạo **những gì bác sĩ X-quang hay ghi** (đáp án là nhãn NLP), không phải chẩn đoán.
- PPV thấp (ví dụ 0,06) nghĩa là phần lớn lần báo "nghi bệnh" thì bác sĩ không ghi bệnh đó. Gợi ý chỉ là **chỗ cần xem lại**.

## B. Tái tạo `demo_0100.png`

### B0. Cài thư viện

Ô này cài vào **đúng môi trường của kernel** đang chạy notebook (`%pip`, không phải `!pip`):
- thư viện của pipeline (`pipeline/requirements.txt`);
- `nibabel`, mà `ai/segment.py` cần nhưng file trên không có.

Chỉ cần chạy một lần.

**Chạy trên máy:** notebook tự tìm gốc repo từ thư mục đang mở.

**Chạy trên Colab** (trên web, hoặc kernel Colab trong VS Code): máy ảo Colab trống, nên ô này `git clone` repo (public) vào thư mục hiện tại. Lưu ý:
- Repo không có ảnh MRI, nên bước B1 sẽ hỏi upload file `0100.nii.gz`.
- Trên Colab, mọi lệnh `%pip` đều kèm ràng buộc `PIP_C`: giữ nguyên các phiên bản mà gói `google-colab` ghim cứng (pandas, requests…). Nếu không, pip có thể nâng pandas lên 3.x và làm hỏng Colab.
- Nếu Colab báo cần *Restart session* sau khi cài (ví dụ pandas vừa bị hạ về đúng bản của Colab), hãy restart rồi chạy lại từ ô này.
- Colab có GPU, nên có thể đặt `RUN_SEGMENT = True`. Lần đầu sẽ tải khoảng 4 GB trọng số mô hình.

TotalSpineSeg (kéo theo PyTorch, khá nặng) **không** cài ở đây. Ô B2 tự cài khi `RUN_SEGMENT = True` mà chưa có.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/hieunghuu/mmis-gr8-lowbackpain"

# gốc repo = thư mục cha gần nhất có cả ai/ và pipeline/src/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "ai").is_dir() and (p / "pipeline" / "src").is_dir()), None)
if ROOT is None and IN_COLAB:          # máy ảo Colab trống -> clone repo
    ROOT = Path.cwd() / "mmis-gr8-lowbackpain"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
assert ROOT, "Không thấy gốc repo: mở notebook từ trong repo, hoặc chạy trên Colab"
print("Gốc repo:", ROOT)

PIP_C = ""   # ràng buộc cho mọi lệnh %pip trong notebook
if IN_COLAB:   # google-colab ghim cứng pandas, requests... -> cấm pip nâng các gói này (nâng là hỏng Colab)
    from importlib.metadata import requires
    pins = [r.split(";")[0].strip() for r in requires("google-colab") or [] if "==" in r]
    Path("/tmp/colab_constraints.txt").write_text("\n".join(pins))
    PIP_C = "-c /tmp/colab_constraints.txt"
%pip install -q {PIP_C} -r "{ROOT / 'pipeline' / 'requirements.txt'}" nibabel

### Cấu hình

In [ ]:
import json, os, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk

# ROOT lấy từ ô B0
sys.path[:0] = [str(ROOT / "ai"), str(ROOT / "pipeline" / "src")]

PID = 100             # mã bệnh nhân trong bộ dữ liệu
RUN_SEGMENT = False   # True: chạy TotalSpineSeg thật (~18 s GPU, ~14 phút CPU)
DEMO = ROOT / "ai" / "demo"
IMAGE = ROOT / "pipeline" / "outputs" / "nifti" / "sag" / f"{PID:04d}.nii.gz"   # cùng chỗ s02 ghi
WORK = ROOT / "pipeline" / "outputs" / "cache" / f"demo_{PID:04d}"              # đã gitignore
WORK.mkdir(parents=True, exist_ok=True)

### B1. Ảnh sagittal T2 (bước 1 của pipeline)

Nếu đã chạy `s02_select_convert.py` thì ảnh có sẵn. Nếu chưa, chỉ chuyển **một** bệnh nhân:
- lấy series đã chọn trong `series_selected.csv` (có trong Git);
- dựng ảnh bằng chính `build_volume()` của s02.

Cách này không cần `dicom_index.csv` (24 MB, không có trong Git).

**Trên Colab** không có DICOM (12,5 GB). Ô này mở hộp chọn file, để upload `0100.nii.gz`: trên máy là file `pipeline/outputs/nifti/sag/0100.nii.gz`, tạo bằng s02 hoặc bằng chính notebook này khi chạy local.

Hộp upload **chỉ chạy trên Colab trong trình duyệt**. Kernel Colab trong VS Code báo *"Upload widget is only available when the cell has been executed in the current browser session"*. Vì vậy hãy mở notebook ở colab.research.google.com: *File → Upload notebook*, chọn file này. Cách khác: kéo thả `0100.nii.gz` vào thư mục `pipeline/outputs/nifti/sag/` của repo đã clone (bảng *Files* của Colab), rồi chạy lại ô.

In [ ]:
from common import MRI_DIR, T_DICOM
from dicom.s02_select_convert import build_volume

if not IMAGE.exists():
    sel = pd.read_csv(T_DICOM / "series_selected.csv").set_index("patient_id").loc[PID]
    series_dir = MRI_DIR / f"{PID:04d}" / sel.study_dir / sel.sag_series
    IMAGE.parent.mkdir(parents=True, exist_ok=True)
    if series_dir.is_dir():
        files = sorted(f for f in series_dir.iterdir() if f.is_file())
        # shortcut: lấy mọi file trong series; s02 còn bỏ lát khác hướng khi series có > 1 hướng (hiếm) -> khi đó chạy s02
        vol, qc = build_volume(files)
        sitk.WriteImage(vol, str(IMAGE), useCompression=True)
        print(f"Đã tạo {IMAGE.name} từ {sel.sag_series}: {qc['n_slices']} lát, pixel {qc['sp_x']:.2f} mm, lát {qc['sp_z']:.1f} mm")
    elif IN_COLAB:
        from google.colab import files as colab_files
        print(f"Không có DICOM trên Colab -> chọn file {IMAGE.name} để upload")
        up = colab_files.upload()
        assert len(up) == 1, "Upload đúng 1 file .nii.gz"
        IMAGE.write_bytes(next(iter(up.values())))
    else:
        raise FileNotFoundError(f"Không có {IMAGE} lẫn DICOM {series_dir}: chạy s02 hoặc tải dữ liệu Mendeley")
im = sitk.ReadImage(str(IMAGE))
print("Ảnh:", im.GetSize(), "voxel", tuple(round(s, 2) for s in im.GetSpacing()), "mm")

### B2. Phân đoạn – `ai/segment.py`

`RUN_SEGMENT = True` thì gọi `segment()` thật, đúng như worker của BE:
- chưa có CLI `totalspineseg` thì ô này tự cài `totalspineseg[nnunetv2]`;
- weights để ở `TOTALSPINESEG_DATA`, lần đầu tự tải khoảng 4 GB;
- trên Colab nên chọn *Runtime → T4 GPU*.

Mặc định dùng mask có sẵn, theo thứ tự:
1. `ai/demo/0100_seg.nii.gz` – mask tạo bằng `segment()` trên CPU, khớp hoàn toàn (Dice 1,000) với mask chạy trên Kaggle;
2. mask 515 ca của pipeline (`python src/s00_shared_data.py --fetch`) – dùng được cho bệnh nhân khác.

In [ ]:
import shutil
import segment as seg
from segment import segment

if RUN_SEGMENT:
    # TSS_CMD = CLI cạnh python của kernel, không có thì tên trần (tìm theo PATH, vd /usr/local/bin trên Colab)
    if not shutil.which(seg.TSS_CMD[0]):
        %pip install -q {PIP_C} "totalspineseg[nnunetv2]" "kornia<0.8"
    os.environ.setdefault("TOTALSPINESEG_DATA", str(ROOT / "pipeline" / ".tss_data"))
    r = segment(IMAGE, WORK / "output", timeout=3600)
    MASK = r.mask_path
    print(f"segment(): {r.duration_seconds} s")
else:
    MASK = next((p for p in [DEMO / f"{PID:04d}_seg.nii.gz",
                             ROOT / "pipeline" / "outputs" / "tss" / "sag" / "step2_output" / f"{PID:04d}.nii.gz"]
                 if p.exists()), None)
    assert MASK, "Không có mask sẵn cho bệnh nhân này: chạy s00 --fetch hoặc đặt RUN_SEGMENT = True"
labels = np.unique(sitk.GetArrayViewFromImage(sitk.ReadImage(str(MASK))))
print("Mask:", MASK.relative_to(ROOT), "| nhãn:", [int(v) for v in labels if v])

### B3. Phân tích – `ai/analyze.py`

Đây chính là `result.analysis` mà BE trả cho FE. Với BN 100, kết quả phải trùng [`analysis_0100.json`](analysis_0100.json) đã commit.

In [ ]:
from analyze import analyze

a = analyze(IMAGE, MASK)
print("QC:", "đạt" if a["qc"]["ok"] else a["qc"]["warnings"], "| góc ưỡn:", a["lordosis_deg"], "độ")
for lv in a["levels"]:
    print(f"  {lv['level']:7s} {len(lv['suggestions'])} gợi ý", "(thiếu: " + lv["missing"] + ")" if lv["missing"] else "")

ref = DEMO / f"analysis_{PID:04d}.json"
if ref.exists():
    print("Trùng với", ref.name + ":", json.loads(json.dumps(a)) == json.loads(ref.read_text(encoding="utf-8")))

### B4. Vẽ hình

`analyze()` không trả tọa độ, nên đo lại bằng `measure_case(keep_pts=True)` của s09 để lấy:
- **lát giữa** `k_mid` – lát có nhiều voxel ống sống nhất;
- **tâm và trục** từng đĩa, để đặt nhãn tầng (màu đỏ nếu tầng có gợi ý).

Hàm `to_pixel()` của s09 đổi tọa độ mm sang pixel.

Chữ "(z)" sau ngưỡng: quy tắc dùng ngưỡng z chung, nên số đo được so **sau khi chuẩn hóa z trong tầng**, không so trực tiếp giá trị mm.

In [ ]:
from measurements.s09_level_features import CANAL, measure_case, to_pixel

o = measure_case(MASK, IMAGE, keep_pts=True)
k, I, M, img = o["k_mid"], o["I"], o["M"], o["img"]
hits = {lv["level"]: lv["suggestions"] for lv in a["levels"]}


def color_of(lab):
    if lab in CANAL:
        return (0.35, 0.6, 0.95)        # tủy + ống sống
    if 41 <= lab <= 50:
        return (0.6, 0.45, 0.9)         # đốt L1-L5, L6, xương cùng
    if 91 <= lab <= 100:
        return (0.2, 0.75, 0.8)         # đĩa T12-L1 .. L5-S
    return (0.9, 0.3, 0.25) if lab < 60 else (0.95, 0.85, 0.2)   # đốt / đĩa cổ - ngực


overlay = np.zeros((*M[k].shape, 4))
for lab in np.unique(M[k]):
    if lab:
        overlay[M[k] == lab] = (*color_of(int(lab)), 0.45)

fig, (ax, tx) = plt.subplots(1, 2, figsize=(15, 7.5), gridspec_kw={"width_ratios": [1, 1.25]})
ax.imshow(I[k], cmap="gray", vmax=np.percentile(I[k], 99.5))
ax.imshow(overlay)
for lv, p in o["pts"].items():
    col = "red" if hits.get(lv) else "lime"
    a0 = to_pixel(img, k, *(p["c"] - p["e1"] * p["L"] * 1.3))   # -e1 = phía trước đĩa
    a1 = to_pixel(img, k, *p["c"])
    ax.plot([a0[0], a1[0]], [a0[1], a1[1]], color=col, lw=1)
    ax.text(a0[0] - 3, a0[1], lv, color=col, fontsize=8, ha="right", va="center")
ax.set_title(f"BN {PID:04d} - lát giữa {k}: ảnh T2 + mask (đỏ = có gợi ý)")
ax.axis("off")

lines = []
if not a["qc"]["ok"]:
    lines += ["QC không đạt - không gợi ý:"] + [f"  ! {w['message']}" for w in a["qc"]["warnings"]] + [""]
for lv in a["levels"]:
    lines.append(lv["level"] + (f"  (thiếu: {lv['missing']})" if lv["missing"] else ""))
    lines += [f"   • {s['label_vi']} ({s['confidence']}, PPV {s['ppv']})  {s['measure_vi']} {s['value']} "
              f"{s['rule']} {s['threshold']}" + (" (z)" if s["threshold_unit"] == "z" else "")
              + (f"\n        ↳ {s['literature']}" if s.get("literature") else "")
              for s in lv["suggestions"]] or ["   – không có gợi ý"]
lines += ["", a["disclaimer"]]
tx.text(0, 1, "\n".join(lines), va="top", fontsize=9.5, linespacing=1.5)
tx.set_title("analyze() - gợi ý theo tầng")
tx.axis("off")
fig.tight_layout()
fig.savefig(WORK / f"demo_{PID:04d}.png", dpi=110)
print("Đã lưu", (WORK / f"demo_{PID:04d}.png").relative_to(ROOT))
plt.show()